# Calculation of Redox Potentials

## Step 1: Import Dependencies

We begin by importing all required Python modules for this tutorial. These include:

- **ORCA Python Interface (OPI)**: for setting up, running, and parsing ORCA quantum chemistry calculations.
- **py3Dmol**: for interactive 3D visualization of molecular structures directly in the notebook.

> **Note:** We additionally import modules for visualization/plotting like `py3Dmol`. For this, it might be necessary to install `py3Dmol` into your OPI `venv` (e.g., by activating the `.venv` and using `uv pip install py3Dmol`).

In [ ]:
# > Import pathlib for directory handling
from pathlib import Path
import shutil

# > OPI imports for performing ORCA calculations and reading output
from opi.core import Calculator
from opi.input.structures.structure import Structure
from opi.input.simple_keywords import Dft, Task, SolvationModel, Solvent, BasisSet
from opi.input.simple_keywords import SimpleKeyword
from opi.output.core import Output
from opi.utils.units import AU_TO_EV

# > Import py3Dmol for 3D molecular visualization
import py3Dmol

## Step 2: Define Working Directory

All actual calculations will be performed in a subfolder `redox`.

In [2]:
# > Calculation is performed in `fock_matrix_diagonalization`
working_dir = Path("redox")
# > The `working_dir`is automatically (re-)created
# > shutil.rmtree(working_dir, ignore_errors=True)
#working_dir.mkdir()

## Step 3: Structures




In [3]:
# > define cartesian coordinates as python string
xyz_data = """\
18

  C   -1.80377999852609     -0.73292620376751      0.04181544005301
  C   -0.61588151024248     -1.42269252621680      0.07308338407509
  C   0.60102012281758     -0.74158181279075      0.02782441193017
  C   1.87351658324421     -1.48040266835269      0.02075519867282
  H   2.56861940885543     -1.10165150390611      0.77259229127910
  H   1.72964586949912     -2.54367411207318      0.17572594022257
  H   2.38954643395664     -1.34698664280123     -0.93597686801328
  C   0.59346685657922      0.72513038162464     -0.03579566129783
  C   1.85826995742051      1.47681615025761     -0.03597531187196
  H   2.55807303054273      1.09975187414010     -0.78384890645264
  H   1.70364833211120      2.53754141483307     -0.19800664432315
  H   2.37445543122131      1.35576128116507      0.92261502495743
  C   -0.63058712516407      1.39472488384039     -0.07135956517472
  C   -1.81121541048370      0.69367276405033     -0.03044333637106
  H   -2.75742786007925      1.21553866770527     -0.05051154184134
  H   -0.64251498639640      2.47363594653280     -0.12318703397596
  H   -0.61756731131841     -2.50166102357026      0.12484276285091
  H   -2.74447782403755     -1.26435687067076      0.06961041528083\n
"""

xyz_data_ferrocene ="""
21

Fe    0.0001484   -0.0000700    0.0000003 
C     1.0879404   -0.5584340   -1.6408152 
C     0.8684239    0.8531637   -1.6447701 
C    -0.5419161    1.0805933   -1.6508517 
C    -1.1939660   -0.1904276   -1.6506665 
C    -0.1866951   -1.2033615   -1.6444747 
C     0.8546514    0.8608406    1.6479739 
C     1.0746054   -0.5506850    1.6521854 
C    -0.1998353   -1.1959999    1.6482845 
C    -1.2074202   -0.1833808    1.6416696 
C    -0.5557567    1.0878358    1.6414621 
H    -0.3716963   -2.2636658    1.6237489 
H     2.0370001   -1.0440704    1.6312849 
H    -1.0302386    2.0455425   -1.6287499 
H    -2.2625677   -0.3566696   -1.6282547 
H    -0.3590964   -2.2708566   -1.6165383 
H     2.0499711   -1.0519947   -1.6097614 
H     1.6350810    1.6156966   -1.6171011 
H    -1.0441591    2.0525164    1.6109836 
H     1.6212807    1.6234981    1.6231643 
H    -2.2757539   -0.3500721    1.6112240 
"""

xyz_data_crcp2 ="""
21
CrCp2 0 3
Cr    0.0002776   -0.0375715   -0.0003458 
C     0.0091447    1.2163607    1.6907876 
C    -1.1485372    0.3664982    1.7678614 
C    -0.6950953   -0.9784911    1.8856174 
C     0.7196981   -0.9766377    1.8796584 
C     1.1687471    0.3688367    1.7579480 
H     0.0081294    2.2961817    1.6424823 
H    -2.1804165    0.6893356    1.7496586 
H    -1.3245235   -1.8584374    1.9153038 
H     1.3517599   -1.8549246    1.9026928 
H     2.1997809    0.6936377    1.7283524 
C    -1.1698542    0.3684104   -1.7546014 
C    -0.0096383    1.2159447   -1.6917836 
C    -0.7207104   -0.9774547   -1.8775486 
H    -2.2007823    0.6933168   -1.7235511 
C     1.1473778    0.3660570   -1.7727959 
H    -0.0082056    2.2957764   -1.6437382 
C     0.6940490   -0.9785958   -1.8887251 
H    -1.3524785   -1.8559603   -1.8991746 
H     2.1793487    0.6887843   -1.7562137 
H     1.3237532   -1.8583385   -1.9194144 
"""

# > Visualize the input structure
view = py3Dmol.view(width=400, height=400)
view.addModel(xyz_data, 'xyz')
view.setStyle({}, {'stick': {}, 'sphere': {'scale': 0.3}})
view.zoomTo()
view.show()

# > Read the structure into object
structure = Structure.from_xyz_block(xyz_data)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Step 4: ORCA Calculations

In [ ]:
def run_calc(basename : str, working_dir: Path, sk_list: list[SimpleKeyword], structure: Structure, ncores: int = 4) -> Output:
    """Perform a calculation and get the output"""
    # > Set up a Calculator object, the basename for the ORCA calculation is also set
    calc = Calculator(basename=basename, working_dir=working_dir)
    # > Assign structure to calculator
    calc.structure = structure

    # > Use simple keywords in calculator
    calc.input.add_simple_keywords(*sk_list)

    # > Define number of CPUs for the calcualtion
    calc.input.ncores = ncores # > CPUs for this ORCA run (default: 1)

    # > perform the calculation
    calc.write_and_run()

    # > get the output
    output = calc.get_output()

    # > Check for normal termination, SCF, and if requested geometry optimization
    if not output.terminated_normally():
        raise RuntimeError(f"ORCA calculation failed! Path: {working_dir}/{basename}")
    if not output.scf_converged():
        raise RuntimeError(f"ORCA SCF did not converge! Path: {working_dir}/{basename}!")
    if Task.OPT in sk_list and not output.geometry_optimization_converged():
        raise RuntimeError(f"ORCA geometry optimization did not converge! Path: {working_dir}/{basename}!")
    
    # > Parse the output 
    output.parse()

    # > return the output
    return output

In [ ]:
def run_redox(basename : str, working_dir: Path, structure: Structure, ncores: int = 4) -> tuple[Output, Output]:
    # Define a level of theory: we use r2SCAN-3c for geometry optimization
    opt_sk_list = [
        Dft.R2SCAN_3C, # > r2SCAN-3c method (Comes with a predefined basis set)
        Task.OPT, # > Perform the geometry optimization
        Task.FREQ, # > After geometry optimization perform a frequency calculation
        SolvationModel.CPCM(Solvent.ACETONITRILE) # CPCM solvation model
        # > Note that the order of OPT and FREQ does not matter! FREQ is always performed after OPT! 
    ]

    # > Calculate neutral species
    output_neutral = run_calc(basename=f"{basename}_neutral", working_dir=working_dir, sk_list=opt_sk_list, structure=structure, ncores=ncores)

    # > Calculate charged species
    # > Adjust the charge
    structure.charge = 1
    # > Adjust the electron-spin multiplicity
    structure.set_ls_multiplicity() # equal to structure.multiplicity = 2
    output_cation = run_calc(basename=f"{basename}_cation", working_dir=working_dir, sk_list=opt_sk_list, structure=structure, ncores=ncores)

    # > return outputs    
    return output_cation, output_neutral

output_cation, output_neutral = run_redox("dimethylbenzene", working_dir=working_dir, structure=structure)

# Process the results

In [6]:
e_redox_abs = (output_cation.get_free_energy() - output_neutral.get_free_energy()) * AU_TO_EV
print(f"The calculated absolute redox potential is {e_redox_abs:.2f} V")
e_redox_rel = e_redox_abs - 4.422
print(f"The calculated redox potential is {e_redox_rel:.2f} V")
print("The experimental redox potential is 2.22 V vs SCE")

The calculated absolute redox potential is 6.17 V
The calculated redox potential is 1.75 V
The experimental redox potential is 2.22 V vs SCE


# Look at the spin density of the cation

In [9]:
# > to keep the size of this notebook small
cube_output = output_neutral.plot_spin_density(resolution=30)
cube_data = cube_output.cube

view = py3Dmol.view(width=500, height=500)
view.addModel(xyz_data, "xyz")
view.setStyle({'stick': {'radius': 0.1}, 'sphere': {'scale': 0.2}})
view.addVolumetricData(cube_data, "cube", {"isoval": 0.03, "color": "green", "opacity": 0.8})
view.addVolumetricData(cube_data, "cube", {"isoval": -0.03, "color": "yellow", "opacity": 0.8})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.